# ── 0. 配置区 ──────────────────────────────────────────────

In [ ]:
# ── 0. 配置区 ──────────────────────────────────────────────
from pathlib import Path

# FiftyOne 数据集名称（用于绘制 PR 曲线）
PR_DATASET_NAME = "sahi_null_v3_ms1_0726-0809_11_ok"
PR_PRED_FIELD   = "small_slices_yolo11x_20pct_null_images_add_rawData_batch_8_final"
PR_GT_FIELD     = "ground_truth"
PR_CLASS        = "swd"

# 无 GT 导出配置（来自 37 原有代码）
VERSION_NOGT    = "sahi_null_run_rawData_v4"
OUT_DIR_NOGT    = Path("./_pred_exports2")
MODEL_NAMES_NOGT = [
    "yolo11n_20pct_null_images_add_rawData_batch_8_final.pt",
]
FILENAME_YEAR   = 2024
TIME_FREQ       = "D"
PLOT_BY_FOCUS   = False

OUT_DIR_NOGT.mkdir(parents=True, exist_ok=True)

# ── 1. 初始化日志 ──────────────────────────────────────────

In [ ]:
import logging
import ipynbname

log_file_name = ipynbname.name() + ".log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_file_name, encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger(__name__)
logger.info("============ Notebook logging 初始化完成 ============")

# ── 2. Step 1：绘制 PR 曲线 ─────────────────────────────

In [ ]:
# ── Step 1：加载 dataset，计算 mAP，绘制 PR 曲线 ───────────
# 输入：PR_DATASET_NAME, PR_PRED_FIELD
# 输出：matplotlib PR 曲线图

import numpy as np
import matplotlib.pyplot as plt
import fiftyone as fo

logger.info("Step 1 开始：计算 mAP + 绘制 PR 曲线")

try:
    dataset = fo.load_dataset(PR_DATASET_NAME)
    results = dataset.evaluate_detections(
        PR_PRED_FIELD, gt_field=PR_GT_FIELD, method="coco", compute_mAP=True
    )
    logger.info(f"mAP={results.mAP():.4f}, classes={results.classes}")

    # ── PR 曲线（matplotlib backend）
    plot = results.plot_pr_curves(backend="matplotlib", classes=[PR_CLASS])
    plt.show()
except Exception as e:
    logger.error(f"绘制 PR 曲线失败: {e}")

logger.info("Step 1 完成：PR 曲线绘制结束")

# ── 3. Step 2：F1 vs Confidence（IoU=0.5）─────────────

In [ ]:
# ── Step 2：绘制 F1 vs Confidence（IoU=0.5）───────────────
# 输入：results（Step 1 产出）
# 输出：matplotlib F1-threshold 曲线

logger.info("Step 2 开始：F1 vs Confidence（IoU=0.5）")

try:
    class_idx = np.where(results.classes == PR_CLASS)[0][0]
    iou_list  = np.array(getattr(results, "iou_threshs", None) or getattr(results, "iou_thresholds", []))
    iou_idx   = np.argmin(np.abs(iou_list - 0.5))
    print(f"Using IoU threshold: {iou_list[iou_idx]}")

    P   = results.precision[iou_idx, class_idx, :]
    R   = results.recall
    thr = results.thresholds[iou_idx, class_idx, :]
    F1  = 2 * P * R / (P + R + 1e-8)

    plt.figure()
    plt.plot(thr, F1)
    plt.xlabel("Confidence Threshold")
    plt.ylabel("F1 Score")
    plt.title(f"F1 vs Confidence ({PR_CLASS}, IoU={iou_list[iou_idx]:.2f})")
    plt.grid(True)
    plt.show()
    logger.info(f"Best F1={np.max(F1):.4f} at threshold={thr[np.argmax(F1)]:.4f}")
except Exception as e:
    logger.error(f"绘制 F1-threshold 曲线失败: {e}")

logger.info("Step 2 完成")

# ── 4. Step 3：F1@[0.5:0.95] vs Confidence ──────────

In [ ]:
# ── Step 3：绘制 F1@[0.5:0.95] vs Confidence ──────────────
# 输入：results（Step 1 产出）
# 输出：matplotlib 曲线（COCO style mean across IoU）

logger.info("Step 3 开始：F1@[0.5:0.95] vs Confidence")

try:
    P_all   = results.precision[:, class_idx, :]
    thr_all = results.thresholds[:, class_idx, :]
    F1_all  = np.array([2 * P_all[i] * R / (P_all[i] + R + 1e-8) for i in range(len(iou_list))])
    F1_mean = np.nanmean(F1_all, axis=0)
    thr_mean = np.nanmean(thr_all, axis=0)

    plt.figure()
    plt.plot(thr_mean, F1_mean)
    plt.xlabel("Confidence Threshold")
    plt.ylabel("F1 Score")
    plt.title(f"F1@[0.5:0.95] vs Confidence ({PR_CLASS})")
    plt.grid(True)
    plt.show()
    logger.info(f"Best F1@[0.5:0.95]={np.max(F1_mean):.4f} at threshold={thr_mean[np.argmax(F1_mean)]:.4f}")
except Exception as e:
    logger.error(f"绘制 F1@[0.5:0.95] 曲线失败: {e}")

logger.info("Step 3 完成")

# ── 5. Step 4：无 GT 数据导出 Excel + 时序图 ───────────

In [ ]:
# ── Step 4：从无 GT 的 FiftyOne datasets 导出 Excel + 时序图 ─
# 输入：VERSION_NOGT 前缀的 datasets
# 输出：pred_summary / pred_per_image / pred_timeseries Excel

from __future__ import annotations
import re
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from datetime import datetime

import numpy as np
import pandas as pd
import fiftyone as fo
import matplotlib.pyplot as plt

logger.info("Step 4 开始：无 GT 数据导出")


def model_tag_from_name(name: str) -> str:
    return Path(name).stem


PRED_FIELDS = {model_tag_from_name(n): f"pred_{model_tag_from_name(n)}" for n in MODEL_NAMES_NOGT}
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_SUMMARY_XLSX   = OUT_DIR_NOGT / f"pred_summary__{VERSION_NOGT}__{stamp}.xlsx"
OUT_PER_IMAGE_XLSX = OUT_DIR_NOGT / f"pred_per_image__{VERSION_NOGT}__{stamp}.xlsx"
OUT_TS_XLSX        = OUT_DIR_NOGT / f"pred_timeseries__{VERSION_NOGT}__{stamp}.xlsx"

FN_RE = re.compile(r"^(?P<mmdd>\d{4})_(?P<hhmm>\d{4})_(?P<focus>\d+)\.(jpg|jpeg|png)$", re.IGNORECASE)


def safe_stats(confs: List[float]) -> Dict[str, float]:
    if not confs:
        return {"conf_max": np.nan, "conf_mean": np.nan, "conf_p50": np.nan, "conf_p90": np.nan}
    arr = np.asarray(confs, dtype=float)
    return {"conf_max": float(np.nanmax(arr)), "conf_mean": float(np.nanmean(arr)),
            "conf_p50": float(np.nanpercentile(arr, 50)), "conf_p90": float(np.nanpercentile(arr, 90))}


def get_detections(sample: fo.Sample, pred_field: str) -> list:
    if not sample.has_field(pred_field):
        return []
    val = sample[pred_field]
    if val is None:
        return []
    return getattr(val, "detections", None) or []


def parse_time_focus(filename: str, year: int) -> Tuple[Optional[datetime], Optional[int], Optional[int]]:
    m = FN_RE.match(filename)
    if not m:
        logger.warning(f"文件名格式不符: {filename}")
        return None, None, None
    try:
        dt = datetime(year, int(m.group("mmdd")[:2]), int(m.group("mmdd")[2:]),
                      int(m.group("hhmm")[:2]), int(m.group("hhmm")[2:]))
        focus = int(m.group("focus"))
        date_int = year * 10000 + dt.month * 100 + dt.day
        return dt, date_int, focus
    except ValueError as e:
        logger.warning(f"时间解析失败 {filename}: {e}")
        return None, None, None

In [ ]:
def export_two_excels_for_dataset(
    ds: fo.Dataset, pred_fields: Dict[str, str], filename_year: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary_rows: List[Dict[str, Any]] = []
    per_image_rows: List[Dict[str, Any]] = []
    ds_name = ds.name
    n_samples = len(ds)

    for model_tag, pred_field in pred_fields.items():
        if pred_field not in ds.get_field_schema():
            logger.warning(f"{ds_name} missing pred_field={pred_field}, skip")
            continue
        total_pred = int(ds.count(f"{pred_field}.detections"))
        images_with_pred = 0
        confs_all: List[float] = []

        for sample in ds.iter_samples(progress=True):
            fn = Path(sample.filepath).name
            dt, date_int, focus = parse_time_focus(fn, filename_year)
            det_list = get_detections(sample, pred_field)
            n_det = len(det_list)
            if n_det > 0:
                images_with_pred += 1
            confs = [float(d.confidence) for d in det_list if d.confidence is not None]
            confs_all.extend(confs)
            row = {"dataset": ds_name, "sample_id": str(sample.id), "filepath": sample.filepath,
                   "filename": fn, "datetime": dt, "date_int": date_int, "focus": focus,
                   "model_tag": model_tag, "pred_field": pred_field, "pred_count": n_det}
            row.update(safe_stats(confs))
            per_image_rows.append(row)

        s = {"dataset": ds_name, "model_tag": model_tag, "pred_field": pred_field,
             "num_samples": n_samples, "total_pred_count": total_pred,
             "images_with_pred": images_with_pred,
             "pct_images_with_pred": (images_with_pred / n_samples * 100) if n_samples else np.nan,
             "avg_pred_per_image": (total_pred / n_samples) if n_samples else np.nan}
        s.update(safe_stats(confs_all))
        summary_rows.append(s)

    return pd.DataFrame(summary_rows), pd.DataFrame(per_image_rows)


# ── 运行导出 ──────────────────────────────────────────────
all_summary = []
all_per_image = []

ds_names = sorted([n for n in fo.list_datasets() if n.startswith(f"{VERSION_NOGT}_")])
logger.info(f"Found datasets: {len(ds_names)}")
for n in ds_names:
    print(" -", n)

for ds_name in ds_names:
    try:
        ds = fo.load_dataset(ds_name)
        df_sum, df_img = export_two_excels_for_dataset(ds, PRED_FIELDS, FILENAME_YEAR)
        all_summary.append(df_sum)
        all_per_image.append(df_img)
    except Exception as e:
        logger.error(f"导出失败 {ds_name}: {e}")

df_summary   = pd.concat(all_summary,   ignore_index=True) if all_summary   else pd.DataFrame()
df_per_image = pd.concat(all_per_image, ignore_index=True) if all_per_image else pd.DataFrame()

try:
    with pd.ExcelWriter(OUT_SUMMARY_XLSX,   engine="openpyxl") as w: df_summary.to_excel(w,   sheet_name="summary",   index=False)
    with pd.ExcelWriter(OUT_PER_IMAGE_XLSX, engine="openpyxl") as w: df_per_image.to_excel(w, sheet_name="per_image", index=False)
    logger.info(f"已保存: {OUT_SUMMARY_XLSX}, {OUT_PER_IMAGE_XLSX}")
except Exception as e:
    logger.error(f"保存 Excel 失败: {e}")

logger.info("Step 4 完成：无 GT 数据导出结束")